In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: adopted same-S .10-step dual comparison

Run all executes NEXT_MEAN and NEXT_ALLBIT in the same durable Drive task. Both use saved SUM_STEP, original BASE radius1 and the same support/key/message/rho/readout.

MEAN maximizes original A16 (including31)+trueG with raw scalar1:1. ALLBIT minimizes historical .8*mean32 H_.02+.2*H_.01(trueG). Each masks and normalizes its one combined gradient, then takes one .10 step (positive MEAN, negative ALLBIT). No old .5 updater, projection, scan, rollback, extra step or selection.

MEAN exact max ties are undefined. ALLBIT stops for a max tie only when its state loss is active (G<.01); inactive ties do not block payload descent. Zero/undefined gradients retain completed observations and fixed missing endpoint rows. Mu/gamma are ALLBIT loss parameters, never result PASS gates.

Nominal budget: 1load+2gradient decodes/2VJPs+2ordinary decodes;184 forward chunks (92gradient+92ordinary),92 nominal replay. No encode/DiT/native/scheduler/codec. Serial graphs;2 new endpoints,0 independent sources. All32 bits, six hard bits, all22 state correlations/trueG, lost/gained positives and original BASE distances are retained.

Four quality pairs compare each new RGB against saved BASE/S RGB, with181 frame/180 temporal rows, actual peaks and neighborhoods. Missing old RGB is MISSING without replacement decode. Historical26/32 is retained as a saved reference. State weights also differ, so the comparison does not isolate payload aggregation.

60-second committed-state checkpoints and immediate unit persistence support same-Drive Run all resume; completed units skip and report-only completion loads no model. Unfinished autograd graphs do not survive; rework/failure/replay is recorded per attempt and cumulatively. No real cross-Colab recovery claim. No source/version/Git identity gate.

This package is prepared locally; publish its companion before using the default GitHub source URL. GPU/Colab is user-run. Return the whole task folder including failures. No scientific PASS or combined A/B conclusion.


In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-same-point-dual-v1', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'arms': ['NEXT_MEAN', 'NEXT_ALLBIT'], 'step': 0.1, 'device': 'cuda', 'inputs': {'sum_run': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Gradient-Sum/resumable-gradient-sum-v1/run', 'historical_allbit_metrics': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit/resumable-task-v1/run/ALLBIT_TWO/metrics.json'}}
EXPECTED_COST = {'vae_load': 1, 'gradient_decode': 2, 'decoder_vjp': 2, 'ordinary_decode': 2, 'decode': 4, 'gradient_forward_chunks': 92, 'ordinary_chunks': 92, 'total_forward_chunks': 184, 'nominal_replay_chunks': 92, 'encode': 0, 'dit': 0, 'native': 0, 'scheduler': 0, 'codec': 0, 'new_source_endpoints': 2, 'independent_samples': 0, 'new_windows': 176, 'new_chips': 2816, 'new_metrics': 110, 'gradient_windows': 176, 'gradient_chips': 2816, 'gradient_metrics': 110, 'reused_endpoint_views': 2, 'historical_reference_views': 1, 'quality_pairs': 4, 'endpoint_windows': 352, 'endpoint_chips': 5632, 'endpoint_metrics': 220}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Same-Point-Dual')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-same-point-dual-v1'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Same-Point-Dual-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_same_point_dual_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))
        actual = json.loads((RUN_OUTPUT / 'result.json').read_text())
        fixed_slots.update(run_status=actual.get('status'),
            views=actual.get('views', fixed_slots['views']), counts=actual.get('counts'),
            ordinary_chunks=actual.get('ordinary_chunks'))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    views={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('BASE', 'SUM_STEP', 'HISTORICAL_26', 'NEXT_MEAN', 'NEXT_ALLBIT')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed SAME-POINT-DUAL output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_same_point_dual_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_same_point_dual_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for the unfinished fixed model work'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors',
                         'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    environment_receipt = json.loads(probe_path.read_text())
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_same_point_dual_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_SAME_POINT_DUAL', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_same_point_dual_v1 import finalize_interrupted
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] not in ('COMPLETE', 'COMPLETE_WITH_UNDEFINED_ARMS', 'COMPLETE_WITH_MISSING_EVIDENCE'):
        raise RuntimeError('SAME-POINT-DUAL engineering interruption; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status=result['status'], result='run/result.json',
        views=result['views'], counts=result['counts'], ordinary_chunks=result['ordinary_chunks'], units=result['units'], steps=result['steps'], quality=result['quality'])
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_SAME_POINT_DUAL', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
comparison = json.loads((RUN_OUTPUT / 'comparison.json').read_text())
print('Task status:', result['status'], '; not scientific PASS')
print('Budget:', result['expected_cost'])
print('Cumulative calls:', result['counts'])
print('This attempt:', result.get('current_attempt_counts', {}))
print('Replay:', result.get('physical_replay_chunks', {}))
for name, row in comparison['views'].items():
    summary = row.get('summary') or {}
    print(name, row['status'], 'positive=', summary.get('positive_count'),
          'G=', summary.get('state_gap'), 'deepest=', summary.get('deepest_negative_indices'),
          'hard bits=', summary.get('hard_bits'), 'reason=', row.get('reason'))
for name, row in comparison['arms'].items():
    print(name, 'construction=', row['construction'], 'all-bit/state comparisons=', row['comparisons'])
print('Four quality pairs:', comparison['quality'])
print('Failures retained:', result['failure_history'])
print(result['ceiling'])
print('Return whole directory:', OUTPUT)